<a href="https://colab.research.google.com/github/matiasmorgan1/Master-s-by-Research/blob/main/CMEMS_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1️⃣ Install required packages
!pip install xarray netcdf4 plotly pandas cartopy
!pip install geopandas shapely pyproj fiona

# 2️⃣ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3️⃣ Import libraries
import xarray as xr
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer, CRS
from scipy.interpolate import griddata
import geopandas as gpd
from shapely.geometry import Polygon, LineString

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.8 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# Visualise ocean depths near marine-terminating glacier termini.

import numpy as np
import xarray as xr
import plotly.graph_objects as go

# --- Load dataset ---
path = "/content/drive/MyDrive/MRes_OceanData/CMEMS_OceanTemp_1991_2024.nc"
ds = xr.open_dataset(path)

# --- Glacier coordinates ---
glacier_coords = {
    226: (66.401121, -37.597138),
    284: (63.875912, -41.751149),
    303: (62.49107, -43.15229),
    320: (61.639677, -43.136743),
    323: (61.5672, -43.178178),
    245: (65.012344, -41.251558),
    286: (63.852942, -41.81786),
    296: (63.271613, -41.893431),
    233: (65.559332, -40.125435),
    238: (65.234993, -40.706336),
    232: (65.697643, -39.734977),
    300: (62.746836, -43.313508),
    304: (62.370736, -43.051618),
    310: (62.067368, -42.672088),
    270: (64.243946, -41.686908),
    335: (61.209418, -43.313418),
    297: (63.243014, -42.250839),
    298: (62.892129, -42.663693),
    281: (63.902382, -41.356373),
    293: (63.671452, -41.66253),
    224: (66.514517, -36.7617),
    239: (65.194252, -41.11792),
    227: (66.377318, -38.268088),
    258: (64.527037, -40.735331),
    266: (64.370986, -41.555998),
    302: (62.58016, -43.149846),
    330: (61.329731, -43.356673),
}

# --- Compute maximum depth per point ---
valid_mask = np.isfinite(ds["thetao"])
max_depth_time = ds["depth"].where(valid_mask)
max_depth_2d = max_depth_time.max(dim=["depth", "time"])

lon = ds.longitude.values
lat = ds.latitude.values
depth = max_depth_2d.values

# --- Create interactive heatmap ---
fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        x=lon,
        y=lat,
        z=depth,
        colorscale="Plasma",
        zmin=0,
        zmax=900,
        colorbar=dict(title="Maximum Ocean Depth (m)")
    )
)

# --- Add glacier points ---
glon = [v[1] for v in glacier_coords.values()]
glat = [v[0] for v in glacier_coords.values()]
glabels = list(glacier_coords.keys())

fig.add_trace(
    go.Scatter(
        x=glon,
        y=glat,
        mode="markers+text",
        marker=dict(color="red", size=8),
        text=glabels,
        textposition="top right",
        name="Glaciers"
    )
)

# --- Layout ---
fig.update_layout(
    title="Maximum Ocean Depth (CMEMS Data)",
    xaxis_title="Longitude",
    yaxis_title="Latitude",
    width=900,
    height=700
)

fig.show()

# Export per-glacier data to excel

In [ ]:
# ======================================================================
# EXPORT CMEMS DEPTH–TIME ANOMALIES
# • One Excel workbook containing all glaciers
# • One worksheet per glacier
# • Group-specific 5 × 5 grid averages
# • Dates (mm/yyyy) down rows
# • Depth-bin centres across columns
# • Individual group spreadsheets also saved
# ======================================================================

import os
import numpy as np
import pandas as pd
import xarray as xr

# ======================================================================
# 1. ORIGINAL FILE PATHS
# ======================================================================

input_path = (
    "/content/drive/MyDrive/MRes_OceanData/"
    "CMEMS_OceanTemp_1991_2024.nc"
)

input_dir = (
    "/content/drive/MyDrive/MRes_OceanData/"
    "AnomalyFigures/ExcelData_Formatted"
)

output_file = (
    "/content/drive/MyDrive/MRes_OceanData/"
    "CMEMS_AllGlaciers_TemperatureAnomalies.xlsx"
)

# Retain the original group-spreadsheet output directory
export_dir = input_dir
os.makedirs(export_dir, exist_ok=True)

# ======================================================================
# 2. GLACIER GROUPS
# ======================================================================

glacier_groups = {
    "226_224_227": ["226", "224", "227"],
    "233_232": ["233", "232"],
    "239_238_245": ["239", "238", "245"],
    "258": ["258"],
    "266_270": ["266", "270"],
    "281_284_286_293": ["281", "284", "286", "293"],
    "296_297": ["296", "297"],
    "298_300": ["298", "300"],
    "302_303_304": ["302", "303", "304"],
    "310": ["310"],
    "320_323": ["320", "323"],
    "330_335": ["330", "335"],
}

# ======================================================================
# 3. GROUP CENTRES
# Format: (latitude, longitude)
# ======================================================================

center_pixels = {
    "226_224_227": (65.125, -38.125),
    "233_232": (65.125, -38.125),
    "239_238_245": (64.625, -39.625),
    "258": (64.25, -39.75),
    "266_270": (63.875, -39.875),
    "281_284_286_293": (63.75, -39.875),
    "296_297": (62.875, -40.75),
    "298_300": (62.625, -41.125),
    "302_303_304": (62.375, -41.5),
    "310": (62.0, -41.5),
    "320_323": (61.5, -41.5),
    "330_335": (61.25, -41.75),
}

# ======================================================================
# 4. HELPER FUNCTIONS
# Assumes 1D latitude and longitude coordinates
# ======================================================================

def nearest_index(ds, target_lat, target_lon):
    lat = ds["latitude"].values
    lon = ds["longitude"].values

    if lat.ndim != 1 or lon.ndim != 1:
        raise ValueError(
            "Expected 1D latitude and longitude coordinates."
        )

    iy = int(np.abs(lat - target_lat).argmin())

    lon_difference = np.abs(
        (lon - target_lon + 180) % 360 - 180
    )
    ix = int(lon_difference.argmin())

    return iy, ix


def get_5x5(iy, ix, nlat, nlon):
    return [
        (i, j)
        for i in range(max(0, iy - 2), min(nlat, iy + 3))
        for j in range(max(0, ix - 2), min(nlon, ix + 3))
    ]


# ======================================================================
# 5. PROCESS DATASET
# ======================================================================

all_glacier_sheets = {}

with xr.open_dataset(input_path) as ds:

    required = [
        "thetao", "depth", "time", "latitude", "longitude"
    ]
    missing = [v for v in required if v not in ds]

    if missing:
        raise ValueError(f"Missing dataset variables: {missing}")

    if "depth" not in ds["thetao"].dims:
        raise ValueError("'thetao' has no depth dimension.")

    if "time" not in ds["thetao"].dims:
        raise ValueError("'thetao' has no time dimension.")

    # Define 50 m depth bins
    max_depth = float(ds["depth"].max().values)
    upper_edge = (int(np.ceil(max_depth / 50)) + 1) * 50

    depth_bins = np.arange(0, upper_edge + 1, 50)
    bin_centres = (depth_bins[:-1] + depth_bins[1:]) / 2

    depth_values = ds["depth"].values

    for group_name, glacier_ids in glacier_groups.items():

        print(f"\nProcessing group {group_name} ...")

        # --------------------------------------------------------------
        # A. Locate the group centre and select a 5 × 5 grid
        # --------------------------------------------------------------

        centre_lat, centre_lon = center_pixels[group_name]

        iyc, ixc = nearest_index(ds, centre_lat, centre_lon)

        indices = get_5x5(
            iyc, ixc,
            ds.sizes["latitude"],
            ds.sizes["longitude"]
        )

        if len(indices) != 25:
            print(
                f"Warning: only {len(indices)} grid cells "
                f"available for group {group_name}."
            )

        # --------------------------------------------------------------
        # B. Monthly temperature and spatial averaging
        # --------------------------------------------------------------

        da_points = []

        for iy, ix in indices:
            da = ds["thetao"].isel(
                latitude=iy,
                longitude=ix
            )
            da_points.append(
                da.resample(time="MS").mean()
            )

        da_mean = xr.concat(
            da_points, dim="pt"
        ).mean("pt", skipna=True)

        # --------------------------------------------------------------
        # C. Monthly climatology, 1991–2011
        # --------------------------------------------------------------

        baseline = da_mean.sel(
            time=slice("1991-01-01", "2011-12-31")
        )

        if baseline.sizes.get("time", 0) == 0:
            raise ValueError(
                f"No 1991–2011 baseline for {group_name}."
            )

        climatology = (
            baseline.groupby("time.month")
            .mean("time", skipna=True)
        )

        # --------------------------------------------------------------
        # D. Calculate anomalies from 2014 onwards
        # --------------------------------------------------------------

        da_recent = da_mean.sel(
            time=slice("2014-01-01", None)
        )

        if da_recent.sizes.get("time", 0) == 0:
            raise ValueError(
                f"No data from 2014 onwards for {group_name}."
            )

        anomalies = (
            da_recent.groupby("time.month") - climatology
        )

        # --------------------------------------------------------------
        # E. Calculate mean anomaly in each 50 m depth bin
        # --------------------------------------------------------------

        binned = []

        for i in range(len(depth_bins) - 1):

            lower = depth_bins[i]
            upper = depth_bins[i + 1]

            if i == len(depth_bins) - 2:
                mask = (
                    (depth_values >= lower)
                    & (depth_values <= upper)
                )
            else:
                mask = (
                    (depth_values >= lower)
                    & (depth_values < upper)
                )

            layer = anomalies.where(
                xr.DataArray(
                    mask,
                    dims=["depth"],
                    coords={"depth": ds["depth"]}
                ),
                drop=True
            )

            if layer.sizes.get("depth", 0) == 0:
                layer_mean = xr.full_like(
                    anomalies.isel(depth=0, drop=True),
                    np.nan,
                    dtype=float
                )
            else:
                layer_mean = layer.mean(
                    "depth", skipna=True
                )

            layer_mean = layer_mean.expand_dims(
                depth_mid=[bin_centres[i]]
            )
            binned.append(layer_mean)

        da_binned = xr.concat(binned, dim="depth_mid")

        # --------------------------------------------------------------
        # F. Format table: dates in rows, depths in columns
        # --------------------------------------------------------------

        df = da_binned.to_dataframe(
            name="Temperature anomaly"
        ).reset_index()

        pivot = df.pivot(
            index="time",
            columns="depth_mid",
            values="Temperature anomaly"
        ).sort_index()

        pivot.index = pd.to_datetime(
            pivot.index
        ).strftime("%m/%Y")

        pivot.index.name = "Date"

        pivot.columns = [
            f"{float(d):.0f} m" for d in pivot.columns
        ]

        final_df = pivot.reset_index()

        # --------------------------------------------------------------
        # G. Save one spreadsheet per glacier group
        # --------------------------------------------------------------

        group_path = os.path.join(
            export_dir,
            f"{group_name}_TemperatureAnomalies_2014_present.xlsx"
        )

        final_df.to_excel(
            group_path,
            index=False,
            engine="openpyxl"
        )

        print(f"Saved group file: {group_path}")

        # --------------------------------------------------------------
        # H. Copy the group's results to each glacier's sheet
        # --------------------------------------------------------------

        for glacier_id in glacier_ids:
            sheet_name = str(glacier_id)

            all_glacier_sheets[sheet_name] = final_df.copy()

            print(
                f"  Prepared glacier {sheet_name} "
                f"from group {group_name}"
            )

# ======================================================================
# 6. COMBINE ALL GLACIERS INTO ONE EXCEL WORKBOOK
# ======================================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    for glacier_id, df in all_glacier_sheets.items():
        df.to_excel(
            writer,
            sheet_name=glacier_id[:31],
            index=False
        )

print("\nAll glacier-group spreadsheets exported.")
print(f"Combined workbook saved to: {output_file}")
print(f"Total glacier worksheets: {len(all_glacier_sheets)}")